# ISL-Translator: ST-GCN Pre-training on INCLUDE Dataset
This notebook merges the `.npz` keypoints from Part 1 and Part 2, automatically generates the train/val split annotations, and launches the PyTorch training loop across the entire dataset.

In [1]:
!pip install -q "torch>=2.0.0" "torchvision" "torchaudio" "tqdm" "pandas" "pyyaml" mediapipe
print("✅ MediaPipe dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.3 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━ 5.6/10.3 MB 81.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 10.3/10.3 MB 106.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


✅ MediaPipe dependencies installed.


In [2]:
import os
import glob
import json
import random
import shutil
from pathlib import Path

# --- Configuration ---
KAGGLE_INPUT_DIR = "/kaggle/input"
WORKING_DIR = Path("/kaggle/working/data")
TRAIN_DIR = WORKING_DIR / "train"
VAL_DIR = WORKING_DIR / "val"

# Clean previous runs
if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)

TRAIN_DIR.mkdir(parents=True)
VAL_DIR.mkdir(parents=True)

# --- Find all Keypoint Files ---
# We glob through ALL folders in /kaggle/input avoiding the codebase itself
all_npz_files = []
for f in glob.glob(f"{KAGGLE_INPUT_DIR}/**/*.npz", recursive=True):
    if "isl-translator-codebase" not in f: # Don't search inside the codebase folder if it has any for some reason
        all_npz_files.append(f)

print(f"Found {len(all_npz_files)} keypoint .npz files total across all datasets!")
if len(all_npz_files) == 0:
    raise ValueError("No .npz files found! Did you attach the Part 1 and Part 2 datasets?")

# --- Parse Gloss Labels ---
videos = []
for npz_path in all_npz_files:
    filename = os.path.basename(npz_path)
    video_id = filename.replace(".npz", "")
    
    # Typical INCLUDE filename structure: 1._Dog___MVI_3060.npz
    # We want to extract "Dog" as the gloss category
    category_part = video_id.split("___")[0]
    if "_" in category_part and category_part[0].isdigit():
        # E.g. "1._Dog" -> "Dog"
        gloss = category_part.split("_", 1)[1]
    else:
        gloss = category_part
        
    gloss = gloss.upper().replace(".", "").strip()
    videos.append({
        "video_id": video_id,
        "path": npz_path,
        "gloss": gloss
    })

# Verify vocabulary distribution
unique_glosses = set([v['gloss'] for v in videos])
print(f"Parsed {len(unique_glosses)} unique ISL gloss categories.")

# --- Train/Val Split (80/20) ---
random.seed(42)
random.shuffle(videos)
split_idx = int(len(videos) * 0.8)
train_videos = videos[:split_idx]
val_videos = videos[split_idx:]

def process_split(split_data, target_dir, annot_name):
    annotations = []
    for v in split_data:
        # Symlink file into the target directory (instant, uses 0 disk space)
        dst = target_dir / f"{v['video_id']}.npz"
        if not dst.exists():
            os.symlink(v['path'], dst)
            
        # Create JSON annotation
        annotations.append({
            "video_id": v['video_id'],
            "glosses": [v['gloss']],
            "text_english": v['gloss'].lower(),
            "text_hindi": ""
        })
    
    # Save JSON
    with open(WORKING_DIR / annot_name, 'w', encoding='utf-8') as f:
        json.dump(annotations, f, indent=2)
        
    print(f"{annot_name}: {len(annotations)} samples written.")

print("\n--- Generating Symlinks and Annotations ---")
process_split(train_videos, TRAIN_DIR, "train_annotations.json")
process_split(val_videos, VAL_DIR, "val_annotations.json")
print("✅ Dataset preparation complete! Ready for PyTorch loop.")

Found 4289 keypoint .npz files total across all datasets!
Parsed 263 unique ISL gloss categories.

--- Generating Symlinks and Annotations ---
train_annotations.json: 3431 samples written.
val_annotations.json: 858 samples written.
✅ Dataset preparation complete! Ready for PyTorch loop.


In [3]:
# --- Locate the ISL Codebase ---
import sys

CODEBASE_DIR = None
for d in glob.glob("/kaggle/input/*/*"):
    if "isl-translator-codebase" in d or os.path.exists(os.path.join(d, "scripts/train.py")):
        CODEBASE_DIR = d
        break

if not CODEBASE_DIR:
    # Fallback to specifically named folders if glob fails weirdly
    possible_paths = [
        "/kaggle/input/isl-translator-codebase",
        "/kaggle/input/datasets/lastlegend/isl-translator-codebase"
    ]
    for p in possible_paths:
        if os.path.exists(p):
            CODEBASE_DIR = p
            break

if CODEBASE_DIR:
    print(f"✅ ISL codebase found at: {CODEBASE_DIR}")
else:
    raise FileNotFoundError("Could not find the isl-translator-codebase directory in /kaggle/input/. Did you attach it?")

✅ ISL codebase found at: /kaggle/input/datasets/lastlegend/isl-translator-codebase


In [4]:
# --- Launch Training! ---
import subprocess

config_path = os.path.join(CODEBASE_DIR, "configs", "config.yaml")
train_script = os.path.join(CODEBASE_DIR, "scripts", "train.py")

CHECKPOINTS_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

cmd = [
    "python", train_script,
    "--config", config_path,
    "--data_dir", str(WORKING_DIR),  # Point to the auto-generated symlink directory
    "--output_dir", CHECKPOINTS_DIR,
    "--device", "cuda"
]

print("🚀 Launching Training Pipeline...")
print("Command:", " ".join(cmd))

# Run and stream output block-by-block
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")
    
process.wait()
if process.returncode == 0:
    print("🎉 Training Complete! Model saved to /kaggle/working/checkpoints/")
else:
    print("❌ Training failed. Check logs above.")

🚀 Launching Training Pipeline...
Command: python /kaggle/input/datasets/lastlegend/isl-translator-codebase/scripts/train.py --config /kaggle/input/datasets/lastlegend/isl-translator-codebase/configs/config.yaml --data_dir /kaggle/working/data --output_dir /kaggle/working/checkpoints --device cuda


2026-03-11 15:01:44.028668: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered


E0000 00:00:1773241304.426793      52 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773241304.526116      52 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


W0000 00:00:1773241305.442102      52 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773241305.442154      52 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773241305.442157      52 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773241305.442159      52 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.


INFO:__main__:Using GPU: Tesla P100-PCIE-16GB
INFO:__main__:GPU Memory: 17.1 GB
INFO:__main__:Loading datasets...


...

Validating:   0%|          | 0/108 [00:00<?, ?it/s]


Validating:   1%|          | 1/108 [00:07<12:56,  7.25s/it]


Validating:   2%|▏         | 2/108 [00:13<12:12,  6.91s/it]


Validating:   3%|▎         | 3/108 [00:21<12:49,  7.33s/it]


Validating:   4%|▎         | 4/108 [00:27<11:39,  6.72s/it]
